## Gold to Semantic

Publica a camada `semantic` — o **contrato de consumo do dashboard** (Seção 5 do `gold_dimensional_model.md`). O dashboard lê só daqui, nunca da gold atômica. Todas as tabelas cobrem **todas as edições** (coluna `ano_edicao`) e **todos os radares** (coluna `radar`), e respeitam o desenho de cada radar em cada ano: nenhuma tabela tem número fixo de radares, dimensões ou itens.

**Tabelas, não views:** cada objeto é gravado como tabela Delta pronta (recalculada a cada execução, com todas as edições e radares). Assim o dashboard só lê — sem refazer joins, rankings e agregações sobre a gold a cada acesso. Os nomes `vw_*` foram mantidos para não quebrar quem já consome.

| Tabela | Grão | Tela do dashboard |
|---|---|---|
| `vw_radares` | radar | Seletor de radar (nome, descrição, qual é o padrão) |
| `vw_painel_municipio` | radar × município × edição | Visão geral (KPIs, tabela, mapa) e Perfil (cabeçalho e rankings) |
| `vw_score_dimensao` | radar × município × dimensão × edição | Radar, barras por dimensão, Dimensões (ranking e distribuição) |
| `vw_radar_dimensao` | radar × dimensão × edição | Eixos do radar e seletor de dimensão (vem da planilha do especialista) |
| `vw_item_municipio` | radar × município × item × edição | Perfil (os itens do município) |
| `agg_adocao_indicador` | radar × item × UF × edição | % de adoção por item (somas reagregáveis para região/Brasil) |
| `agg_adocao_uf_porte` | radar × item × UF × porte × edição | Perfil (% de adoção na UF e no porte) e Dimensões (adoção por item em qualquer recorte) |
| `agg_imdm_porte_regiao` | radar × região × porte × edição | IMDM por porte e por região |
| `vw_auditoria_lgpd` | município × edição | Bloco LGPD (`mgov32`–`36`); não depende de radar |
| `agg_qualidade_item` | radar × variável do radar × UF × edição | Qualidade dos dados: classes de resposta por variável |

- 1. Pré-requisitos: gold e radares da edição presentes (ledger)
- 2. Gravação das tabelas (sobrescreve; uma view antiga de mesmo nome é removida antes)
- 3. Controles: as tabelas fecham com a gold da edição, radar por radar
- 4. Ledger: linha `semantic` da edição


In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions

## Step 1 - Pré-requisitos

- A gold da edição precisa ter fechado (linha `gold` no ledger) e os radares da edição carregados (linha `radar`)
- As tabelas leem `gold.*` e `config.radar*`; o schema `semantic` é criado pelo `silver_to_gold` (Ajuste 3), recriado aqui por segurança


In [ ]:
from pyspark.sql import functions as F

semantic_schema = "semantic"
G = f"{catalog_name}.{gold_schema}"
C = f"{catalog_name}.config"
S = f"{catalog_name}.{semantic_schema}"

contrato_gold  = contrato_da_edicao(etapa="gold",  ano_edicao=reference_year)
contrato_radar = contrato_da_edicao(etapa="radar", ano_edicao=reference_year)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S} COMMENT 'Camada de consumo do dashboard IMDM (tabelas prontas sobre gold + radares)'")

for t in ["dim_municipio", "dim_indicador", "fact_resposta", "fact_pontos_item", "fact_score_dimensao", "fact_imdm_municipio"]:
    if not spark.catalog.tableExists(f"{G}.{t}"):
        raise RuntimeError(f"Pré-requisito ausente: {G}.{t}. Rode o silver_to_gold da edição {reference_year}.")
radares_cad = sorted(r["radar"] for r in spark.table(f"{C}.radares").select("radar").collect())
print(f"OK: gold {reference_year} ({contrato_gold['linhas']} municípios) | radares {radares_cad} | "
      f"planilha sha256 {contrato_radar['sha256_xlsx'][:12]}…")


## Step 2 - Tabelas

- **Radar:** toda tabela que depende da metodologia tem a coluna `radar`; o app filtra pelo radar escolhido (padrão em `vw_radares`)
- **Ranking:** `rank_*` só entre municípios com IMDM válido, por radar e edição (Brasil, UF e porte); `total_*` acompanha para o "Xº de Y"
- **Status do item** (na variável-âncora): `Sim` (1 ponto), `Parcial` (entre 0 e 1), `Não`, `Salto (0)`, `Não sabe`, `Não informou`, `Não se aplica` (fora do denominador) e `Recusa`
- **Adoção** = Σ pontos / Σ peso × 100 — as somas ficam na tabela para reagregar corretamente por região ou Brasil
- **`agg_adocao_uf_porte`**: as mesmas somas no grão UF × porte, para o app filtrar por região, UF e porte sem ler a `vw_item_municipio` inteira


In [ ]:
geo = "m.nome, m.uf_sigla, m.regiao, m.populacao_total, m.faixa_populacao, m.porte_municipio"

views = {}

views["vw_radares"] = ("Radares IMDM (metodologias) cadastrados pelo especialista; padrao = o que o dashboard abre primeiro.", f"""
SELECT radar, nome, descricao, padrao, radar_sha256
FROM {C}.radares
""")

views["vw_painel_municipio"] = ("Radar × município × edição: IMDM, nível, motivo de nulo e rankings (Brasil, UF, porte).", f"""
WITH base AS (
  SELECT f.radar, m.cod_ibge, m.ano_edicao, {geo},
         f.score_global_imdm, f.nivel_maturidade_mmd, f.motivo_imdm_nulo,
         f.n_dimensoes_imdm, f.n_dimensoes_pontuadas, f.radar_sha256
  FROM {G}.dim_municipio m
  JOIN {G}.fact_imdm_municipio f ON f.cod_ibge = m.cod_ibge AND f.ano_edicao = m.ano_edicao
)
SELECT *,
  CASE WHEN score_global_imdm IS NOT NULL THEN RANK() OVER (PARTITION BY radar, ano_edicao, score_global_imdm IS NULL ORDER BY score_global_imdm DESC) END AS rank_brasil,
  COUNT(score_global_imdm) OVER (PARTITION BY radar, ano_edicao) AS total_brasil,
  CASE WHEN score_global_imdm IS NOT NULL THEN RANK() OVER (PARTITION BY radar, ano_edicao, uf_sigla, score_global_imdm IS NULL ORDER BY score_global_imdm DESC) END AS rank_uf,
  COUNT(score_global_imdm) OVER (PARTITION BY radar, ano_edicao, uf_sigla) AS total_uf,
  CASE WHEN score_global_imdm IS NOT NULL THEN RANK() OVER (PARTITION BY radar, ano_edicao, porte_municipio, score_global_imdm IS NULL ORDER BY score_global_imdm DESC) END AS rank_porte,
  COUNT(score_global_imdm) OVER (PARTITION BY radar, ano_edicao, porte_municipio) AS total_porte
FROM base
""")

views["vw_radar_dimensao"] = ("Dimensões de cada radar × edição (definidas na planilha do especialista): eixos e seletor.", f"""
SELECT radar, edicao AS ano_edicao, dimensao_codigo, dimensao AS dimensao_nome, ordem AS dimensao_ordem,
       peso AS dimensao_peso, lower(entra_no_imdm) = 'sim' AS entra_no_imdm, descricao, radar_sha256
FROM {C}.radar_dimensao
""")

views["vw_score_dimensao"] = ("Radar × município × dimensão × edição: score da dimensão com dados geográficos (radar e barras).", f"""
SELECT s.radar, s.cod_ibge, s.ano_edicao, {geo},
       s.dimensao_codigo, s.dimensao_nome, s.dimensao_ordem, s.dimensao_peso, s.entra_no_imdm,
       s.pontos_obtidos, s.pontos_possiveis, s.score
FROM {G}.fact_score_dimensao s
JOIN {G}.dim_municipio m ON m.cod_ibge = s.cod_ibge AND m.ano_edicao = s.ano_edicao
""")

views["vw_item_municipio"] = ("Radar × município × item × edição: resposta da âncora, pontos e status do item.", f"""
SELECT p.radar, p.cod_ibge, p.ano_edicao, {geo},
       p.dimensao_codigo, d.dimensao AS dimensao_nome, d.ordem AS dimensao_ordem,
       p.item, i.descricao AS item_descricao, p.regra, p.variavel_ancora,
       r.resposta_original, r.classe_preenchimento, p.pontos_obtidos, p.peso_aplicado,
       CASE
         WHEN r.classe_preenchimento = 'Recusa' THEN 'Recusa'
         WHEN p.peso_aplicado IS NULL THEN 'Não se aplica'
         WHEN p.pontos_obtidos >= 1 THEN 'Sim'
         WHEN p.pontos_obtidos > 0 THEN 'Parcial'
         WHEN r.classe_preenchimento = 'Nao_Sabe' THEN 'Não sabe'
         WHEN r.classe_preenchimento = 'Nao_Informou' THEN 'Não informou'
         WHEN r.classe_preenchimento = 'Nao_Aplicavel_Skip' THEN 'Salto (0)'
         ELSE 'Não'
       END AS status_item
FROM {G}.fact_pontos_item p
JOIN {G}.fact_resposta r ON r.cod_ibge = p.cod_ibge AND r.ano_edicao = p.ano_edicao AND r.cod_variable = p.variavel_ancora
JOIN {C}.radar_item i ON i.radar = p.radar AND i.edicao = p.ano_edicao AND i.item = p.item
JOIN {C}.radar_dimensao d ON d.radar = p.radar AND d.edicao = p.ano_edicao AND d.dimensao_codigo = p.dimensao_codigo
JOIN {G}.dim_municipio m ON m.cod_ibge = p.cod_ibge AND m.ano_edicao = p.ano_edicao
""")

views["agg_adocao_indicador"] = ("Radar × item × UF × edição: % de adoção (Σ pontos / Σ peso × 100) com as somas para reagregar.", f"""
SELECT radar, ano_edicao, uf_sigla, regiao, dimensao_codigo, dimensao_nome, dimensao_ordem, item, item_descricao, variavel_ancora,
       COUNT(*) AS municipios,
       COUNT(peso_aplicado) AS municipios_no_denominador,
       SUM(pontos_obtidos) AS soma_pontos,
       SUM(peso_aplicado) AS soma_peso,
       ROUND(SUM(pontos_obtidos) / NULLIF(SUM(peso_aplicado), 0) * 100, 2) AS adocao_pct
FROM {S}.vw_item_municipio
GROUP BY radar, ano_edicao, uf_sigla, regiao, dimensao_codigo, dimensao_nome, dimensao_ordem, item, item_descricao, variavel_ancora
""")

views["agg_adocao_uf_porte"] = ("Radar × item × UF × porte × edição: somas de pontos e peso para a % de adoção em qualquer recorte (app).", f"""
SELECT radar, ano_edicao, uf_sigla, regiao, porte_municipio, dimensao_codigo, dimensao_nome, dimensao_ordem,
       item, item_descricao, variavel_ancora,
       COUNT(*) AS municipios,
       SUM(pontos_obtidos) AS soma_pontos,
       SUM(peso_aplicado) AS soma_peso
FROM {S}.vw_item_municipio
GROUP BY radar, ano_edicao, uf_sigla, regiao, porte_municipio, dimensao_codigo, dimensao_nome, dimensao_ordem,
         item, item_descricao, variavel_ancora
""")

views["agg_imdm_porte_regiao"] = ("Radar × região × porte × edição: municípios, IMDM médio e distribuição por nível (comparação justa).", f"""
SELECT radar, ano_edicao, regiao, porte_municipio,
       COUNT(*) AS municipios,
       COUNT(score_global_imdm) AS municipios_validos,
       ROUND(AVG(score_global_imdm), 2) AS imdm_medio,
       ROUND(PERCENTILE_APPROX(score_global_imdm, 0.5), 2) AS imdm_mediana,
       SUM(score_global_imdm) AS soma_imdm,
       COUNT_IF(nivel_maturidade_mmd LIKE '1-%') AS nivel_1,
       COUNT_IF(nivel_maturidade_mmd LIKE '2-%') AS nivel_2,
       COUNT_IF(nivel_maturidade_mmd LIKE '3-%') AS nivel_3,
       COUNT_IF(nivel_maturidade_mmd LIKE '4-%') AS nivel_4,
       COUNT_IF(nivel_maturidade_mmd LIKE '5-%') AS nivel_5
FROM {S}.vw_painel_municipio
GROUP BY radar, ano_edicao, regiao, porte_municipio
""")

views["vw_auditoria_lgpd"] = ("Município × edição: respostas do bloco LGPD (mgov32–36) e onde fica o responsável (mgov33). NULL em edição sem o bloco. Não depende de radar.", f"""
SELECT m.cod_ibge, m.ano_edicao, {geo},
       MAX(CASE WHEN r.cod_variable = 'mgov32' THEN r.resposta_original END) AS mgov32_responsavel_lgpd,
       MAX(CASE WHEN r.cod_variable = 'mgov33' THEN r.resposta_original END) AS mgov33_local_responsavel,
       MAX(CASE WHEN r.cod_variable = 'mgov34' THEN r.resposta_original END) AS mgov34_cursos_protecao_dados,
       MAX(CASE WHEN r.cod_variable = 'mgov35' THEN r.resposta_original END) AS mgov35_canal_dados_pessoais,
       MAX(CASE WHEN r.cod_variable = 'mgov36' THEN r.resposta_original END) AS mgov36_lei_municipal_lgpd,
       MAX(CASE WHEN r.cod_variable = 'mgov32' THEN r.resposta_original END) = 'Sim' AS tem_responsavel_lgpd
FROM {G}.dim_municipio m
LEFT JOIN {G}.fact_resposta r
  ON r.cod_ibge = m.cod_ibge AND r.ano_edicao = m.ano_edicao
 AND r.cod_variable IN ('mgov32', 'mgov33', 'mgov34', 'mgov35', 'mgov36')
GROUP BY m.cod_ibge, m.ano_edicao, {geo}
""")

views["agg_qualidade_item"] = ("Radar × variável do radar × UF × edição: contagem por classe de resposta (Qualidade dos dados).", f"""
SELECT v.radar, r.ano_edicao, m.uf_sigla, m.regiao, v.dimensao_codigo, v.item, r.cod_variable, v.e_ancora,
       COUNT(*) AS respostas,
       COUNT_IF(r.classe_preenchimento = 'Valida') AS validas,
       COUNT_IF(r.classe_preenchimento = 'Nao_Sabe') AS nao_sabe,
       COUNT_IF(r.classe_preenchimento = 'Nao_Informou') AS nao_informou,
       COUNT_IF(r.classe_preenchimento = 'Nao_Aplicavel_Skip') AS salto_logico,
       COUNT_IF(r.classe_preenchimento = 'Recusa') AS recusa
FROM {G}.fact_resposta r
JOIN {C}.radar_variavel v ON v.edicao = r.ano_edicao AND v.cod_variable = r.cod_variable
JOIN {G}.dim_municipio m ON m.cod_ibge = r.cod_ibge AND m.ano_edicao = r.ano_edicao
GROUP BY v.radar, r.ano_edicao, m.uf_sigla, m.regiao, v.dimensao_codigo, v.item, r.cod_variable, v.e_ancora
""")

def tipo_objeto(nome):
    """'VIEW', 'MANAGED'/'EXTERNAL' ou None se não existe."""
    try:
        return spark.catalog.getTable(f"{S}.{nome}").tableType.upper()
    except Exception:
        try:   # alternativa pelo information_schema do Unity Catalog
            r = spark.sql(f"SELECT table_type FROM {catalog_name}.information_schema.tables "
                          f"WHERE table_schema = '{semantic_schema}' AND table_name = '{nome}'").collect()
            return r[0][0].upper() if r else None
        except Exception:
            return None

# Grava cada objeto como TABELA (o app só lê, sem recalcular nada). A ordem importa:
# as agregações leem a vw_item_municipio / vw_painel_municipio recém-gravadas.
for nome, (comentario, sql) in views.items():
    if tipo_objeto(nome) == "VIEW":          # semantic antiga (views): troca por tabela
        spark.sql(f"DROP VIEW {S}.{nome}")
    (spark.sql(sql).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{S}.{nome}"))
    spark.sql(f"ALTER TABLE {S}.{nome} SET TBLPROPERTIES ('comment' = '{comentario}')")
    print(f"Tabela publicada: {S}.{nome} ({spark.table(f'{S}.{nome}').count():,} linhas)")


## Step 3 - Controles

Os números das tabelas da edição precisam fechar com a gold, **radar por radar** — a semantic não pode criar nem perder município, dimensão ou ponto.


In [ ]:
def da_edicao(nome, radar=None):
    df = spark.table(f"{S}.{nome}").filter(F.col("ano_edicao") == reference_year)
    return df if radar is None else df.filter(F.col("radar") == radar)

def gold_edicao(nome, radar):
    return spark.table(f"{G}.{nome}").filter((F.col("ano_edicao") == reference_year) & (F.col("radar") == radar))

munic = int(contrato_gold["linhas"])
check_igual(sorted(r["radar"] for r in spark.table(f"{S}.vw_radares").collect()), radares_cad, "vw_radares: radares cadastrados")
check_igual(spark.table(f"{S}.vw_radares").where(F.col("padrao")).count(), 1, "vw_radares: exatamente um radar padrão")
check_igual(da_edicao("vw_auditoria_lgpd").count(), munic, f"vw_auditoria_lgpd: {munic} municípios")

resumo_radares = {}
for rad in radares_cad:
    pre = f"[{rad}]"
    pts_gold = gold_edicao("fact_pontos_item", rad)
    n_itens = pts_gold.select("item").distinct().count()
    n_dims  = gold_edicao("fact_score_dimensao", rad).select("dimensao_codigo").distinct().count()
    validos = gold_edicao("fact_imdm_municipio", rad).where(F.col("score_global_imdm").isNotNull()).count()
    resumo_radares[rad] = {"itens": n_itens, "dimensoes": n_dims, "validos": validos}

    painel = da_edicao("vw_painel_municipio", rad)
    check_igual(painel.count(), munic, f"{pre} vw_painel_municipio: {munic} municípios")
    check_igual(painel.agg(F.max("total_brasil")).collect()[0][0], validos, f"{pre} vw_painel_municipio: total do ranking = IMDM válidos")
    check_igual(painel.where(F.col("rank_brasil").isNotNull()).count(), validos, f"{pre} vw_painel_municipio: rank só para IMDM válido")
    check_igual(da_edicao("vw_score_dimensao", rad).count(), munic * n_dims, f"{pre} vw_score_dimensao: {munic} × {n_dims} dimensões")
    check_igual(da_edicao("vw_radar_dimensao", rad).count(), n_dims, f"{pre} vw_radar_dimensao: dimensões do radar = dimensões da gold")
    itens = da_edicao("vw_item_municipio", rad)
    check_igual(itens.count(), munic * n_itens, f"{pre} vw_item_municipio: {munic} × {n_itens} itens")
    soma_gold = pts_gold.agg(F.sum("pontos_obtidos"), F.sum("peso_aplicado")).collect()[0]
    soma_sem  = itens.agg(F.sum("pontos_obtidos"), F.sum("peso_aplicado")).collect()[0]
    check_igual(round(soma_sem[0], 4), round(soma_gold[0], 4), f"{pre} vw_item_municipio: Σ pontos = Σ pontos da fact_pontos_item")
    check_igual(round(soma_sem[1], 4), round(soma_gold[1], 4), f"{pre} vw_item_municipio: Σ peso = Σ peso da fact_pontos_item")
    ado = da_edicao("agg_adocao_indicador", rad).agg(F.sum("soma_pontos"), F.sum("municipios")).collect()[0]
    check_igual(round(ado[0], 4), round(soma_gold[0], 4), f"{pre} agg_adocao_indicador: Σ pontos fecha com a gold")
    check_igual(int(ado[1]), munic * n_itens, f"{pre} agg_adocao_indicador: Σ municípios = municípios × itens")
    ado_up = da_edicao("agg_adocao_uf_porte", rad).agg(F.sum("soma_pontos"), F.sum("soma_peso"), F.sum("municipios")).collect()[0]
    check_igual(round(ado_up[0], 4), round(soma_gold[0], 4), f"{pre} agg_adocao_uf_porte: Σ pontos fecha com a gold")
    check_igual(round(ado_up[1], 4), round(soma_gold[1], 4), f"{pre} agg_adocao_uf_porte: Σ peso fecha com a gold")
    check_igual(int(ado_up[2]), munic * n_itens, f"{pre} agg_adocao_uf_porte: Σ municípios = municípios × itens")
    porte = da_edicao("agg_imdm_porte_regiao", rad).agg(F.sum("municipios"), F.sum("municipios_validos")).collect()[0]
    check_igual((int(porte[0]), int(porte[1])), (munic, validos), f"{pre} agg_imdm_porte_regiao: municípios e válidos fecham")
    n_vars = spark.table(f"{C}.radar_variavel").filter((F.col("radar") == rad) & (F.col("edicao") == reference_year)).count()
    check_igual(int(da_edicao("agg_qualidade_item", rad).agg(F.sum("respostas")).collect()[0][0]), munic * n_vars,
                f"{pre} agg_qualidade_item: respostas = municípios × {n_vars} variáveis do radar")
resumo_validacoes()
display(da_edicao("vw_painel_municipio").orderBy("radar", "rank_brasil").limit(10))


## Step 4 - Ledger

Linha `semantic` × edição: tabelas publicadas e controles (`docs/arquitetura/pipeline_manifest.md`).


In [ ]:
gravar_linha_manifest(
    etapa="semantic",
    ano_edicao=reference_year,
    tabela_destino=";".join(f"{S}.{v}" for v in views),
    sha256_xlsx=contrato_radar["sha256_xlsx"],
    linhas=munic,
    indicadores=int(contrato_radar["linhas"]),   # itens de todos os radares na edição
    detalhes={
        "tabelas_publicadas": ",".join(views),
        "radares": ", ".join(radares_cad),
        "por_radar": " | ".join(f"{r}: {v['dimensoes']} dimensões, {v['itens']} itens, {v['validos']} IMDM válidos"
                                for r, v in resumo_radares.items()),
    },
)
